# Quant Stack — Full System Demo

This notebook demonstrates the **complete pipeline** end-to-end, from raw
data through to a simulated paper trade:

| Stage | Module | Output |
|-------|--------|--------|
| 1. Data | `src.data.synthetic` + `cleaner` | Clean OHLCV DataFrames |
| 2. Features | `src.features.pipeline` | Technical indicators |
| 3. Models | `src.models.classical` + `evaluation` | Walk-forward CV results |
| 4. Portfolio | `src.portfolio.optimiser` + `risk` | Optimised weights |
| 5. Backtest | `src.backtest.engine` + `strategy` | Equity curves & metrics |
| 6. Execution | `src.execution.broker` + `oms` | Paper trade simulation |

> **Runs fully offline** using synthetic data.  Swap the data source to
> `yfinance` for real market data.

## 1. Setup

In [ ]:
%matplotlib inline

import sys
import warnings
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

# --- Project imports (some modules set matplotlib backend to Agg) ---
from src.utils.config import load_config
from src.data.synthetic import generate_multi_asset_data
from src.data.cleaner import DataCleaner
from src.features.pipeline import FeaturePipeline
from src.models.classical import RandomForestModel, GradientBoostingModel
from src.models.evaluation import walk_forward_cv, compare_models, plot_cv_results
from src.portfolio.optimiser import PortfolioOptimiser, equal_weight, inverse_volatility
from src.portfolio.risk import (
    portfolio_returns, sharpe_ratio, risk_summary, correlation_report,
    max_drawdown, value_at_risk, conditional_var, rolling_sharpe,
)
from src.backtest.engine import BacktestEngine
from src.backtest.strategy import (
    MeanReversionStrategy, MomentumStrategy, MACDCrossoverStrategy,
)
from src.execution.broker import PaperBroker, create_broker
from src.execution.oms import OrderManagementSystem

# Restore inline backend AFTER all imports (src modules override to Agg)
try:
    plt.switch_backend("module://matplotlib_inline.backend_inline")
except Exception:
    pass  # Running outside Jupyter — Agg is fine

plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")

config = load_config()
TICKERS = config["universe"]["tickers"]
SEED = config["general"]["random_seed"]

print(f"Universe: {TICKERS}")
print(f"Base currency: {config['general']['base_currency']}")
print(f"Random seed: {SEED}")
print(f"Execution mode: {config['execution']['mode']}")
print(f"Matplotlib backend: {matplotlib.get_backend()}")

---
## 2. Data Pipeline

Generate synthetic OHLCV data for the five-ticker UK large-cap universe
using correlated geometric Brownian motion, then clean it with
`DataCleaner` (fill gaps, repair OHLC consistency, remove outliers).

In production, swap `generate_multi_asset_data` for `YFinanceFetcher` or
load processed Parquet files from `data/processed/`.

In [ ]:
# Generate ~5 years of correlated synthetic data
multi_data = generate_multi_asset_data(TICKERS, days=1260, seed=SEED)

# Clean the data
cleaner = DataCleaner()
clean_data = cleaner.clean_multiple(multi_data)

# Show info for one ticker
demo_ticker = TICKERS[0]
df = clean_data[demo_ticker]
print(f"=== {demo_ticker} ===")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Columns: {list(df.columns)}")
print()
df.head()

In [ ]:
# Plot raw Close prices for all tickers on one chart
fig, ax = plt.subplots(figsize=(14, 5))
for t in TICKERS:
    normalised = clean_data[t]["Close"] / clean_data[t]["Close"].iloc[0] * 100
    ax.plot(normalised, label=t, linewidth=1)

ax.set_title("Normalised Close Prices (base = 100)")
ax.set_ylabel("Price (rebased)")
ax.set_xlabel("Date")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

The `FeaturePipeline` computes SMA (5/10/20/50/200-day), EMA (12/26-day),
RSI-14, MACD with histogram, Bollinger Bands, ATR-14, multi-horizon returns,
and rolling volatility.  Warm-up rows with NaN are dropped automatically.

We use `pipeline.run()` which returns OHLCV **plus** features — the
strategies need both `Close` and indicator columns.

In [ ]:
# Run feature pipeline on all tickers
pipeline = FeaturePipeline(config=config)
enriched = pipeline.run(clean_data)   # dict[ticker, DataFrame with OHLCV + features]

# Show feature names and shapes
feature_names = pipeline.get_feature_names()
print(f"Generated {len(feature_names)} features per ticker")
print(f"Feature columns: {feature_names[:12]}{'...' if len(feature_names) > 12 else ''}")
print()

# Show one ticker's enriched DataFrame shape
demo_feat = enriched[demo_ticker]
print(f"{demo_ticker} enriched shape: {demo_feat.shape}")
print(f"Date range after warm-up: {demo_feat.index.min().date()} to {demo_feat.index.max().date()}")
demo_feat.head(3)

In [ ]:
# Plot Bollinger Bands for one ticker
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Top: Bollinger Bands ---
ax = axes[0]
ax.plot(demo_feat.index, demo_feat["Close"], label="Close", linewidth=1, color="steelblue")
if "bb_upper" in demo_feat.columns and "bb_lower" in demo_feat.columns:
    ax.plot(demo_feat.index, demo_feat["bb_upper"], "--", label="Upper Band", alpha=0.6, color="grey")
    ax.plot(demo_feat.index, demo_feat["bb_lower"], "--", label="Lower Band", alpha=0.6, color="grey")
    if "sma_20" in demo_feat.columns:
        ax.plot(demo_feat.index, demo_feat["sma_20"], ":", label="SMA 20", alpha=0.7, color="orange")
    ax.fill_between(demo_feat.index, demo_feat["bb_lower"], demo_feat["bb_upper"], alpha=0.08, color="steelblue")
ax.set_title(f"Bollinger Bands \u2014 {demo_ticker}")
ax.set_ylabel("Price")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

# --- Bottom: RSI ---
ax = axes[1]
rsi_col = [c for c in demo_feat.columns if c.startswith("rsi_")]
if rsi_col:
    ax.plot(demo_feat.index, demo_feat[rsi_col[0]], linewidth=1, color="purple", label=rsi_col[0])
    ax.axhline(y=70, color="red", linestyle="--", alpha=0.6, label="Overbought (70)")
    ax.axhline(y=30, color="green", linestyle="--", alpha=0.6, label="Oversold (30)")
    ax.fill_between(demo_feat.index, 30, 70, alpha=0.04, color="grey")
ax.set_title(f"RSI \u2014 {demo_ticker}")
ax.set_ylabel("RSI")
ax.set_xlabel("Date")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap (subset for readability)
ohlcv = {"Open", "High", "Low", "Close", "Volume"}
indicator_cols = [c for c in demo_feat.columns if c not in ohlcv]
# Pick at most 15 features for the heatmap
heatmap_cols = indicator_cols[:15]
corr = demo_feat[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            vmin=-1, vmax=1, annot_kws={"size": 7})
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

---
## 4. Model Training

Create a binary classification target (5-day forward return direction),
then compare RandomForest and GradientBoosting via time-series-aware
walk-forward cross-validation.

A gap of 5 trading days between train and test sets prevents
autocorrelation leakage:

```
|---- train (expanding) ----|-- gap --|---- test fold k ----|
```

In [ ]:
# Prepare features and target for one ticker
demo_ohlcv = clean_data[demo_ticker]

# Direction target: 1 if 5-day forward return > 0, else 0
fwd_ret = demo_ohlcv["Close"].pct_change(5).shift(-5)
target = (fwd_ret > 0).astype(int)
target.name = "direction"

# Generate features-only (no OHLCV columns) for modelling
feat_pipeline = FeaturePipeline(config=config)
features_for_model = feat_pipeline.generate(demo_ohlcv)

# Align features and target on the common index
common_idx = features_for_model.index.intersection(target.dropna().index)
X = features_for_model.loc[common_idx]
y = target.loc[common_idx]

# Drop rows with any remaining NaN
mask = X.notna().all(axis=1) & y.notna()
X = X.loc[mask]
y = y.loc[mask]

print(f"Feature matrix X: {X.shape}")
print(f"Target y: {y.shape}  (class balance: {y.mean():.1%} positive)")
print(f"Date range: {X.index.min().date()} to {X.index.max().date()}")

In [ ]:
# Compare RandomForest and GradientBoosting via walk-forward CV
rf = RandomForestModel(name="RandomForest", config=config, target_type="classification")
gb = GradientBoostingModel(name="GradientBoosting", config=config, target_type="classification")

print("Running walk-forward cross-validation (5 folds, gap=5)...")
comparison = compare_models([rf, gb], X, y, n_splits=5, gap=5, min_train_size=252)
print()
comparison

In [ ]:
# Per-fold IC for the best model
best_name = comparison.index[0]
best_model = rf if best_name == "RandomForest" else gb

cv_results = walk_forward_cv(best_model, X, y, n_splits=5, gap=5, min_train_size=252)

fig = plot_cv_results(cv_results)
plt.show()

In [ ]:
# Top 10 feature importances (train on full dataset)
BestClass = RandomForestModel if best_name == "RandomForest" else GradientBoostingModel
fitted = BestClass(name=best_name, config=config, target_type="classification")
fitted.fit(X, y)

importances = fitted.feature_importances()
top_10 = importances.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_10.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title(f"Top 10 Feature Importances \u2014 {best_name}")
ax.set_xlabel("Importance")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 5. Portfolio Optimisation

Compute daily returns across the full universe and run three optimisation
methods:

- **Mean-variance** (via riskfolio-lib, falls back to equal-weight if
  not installed)
- **Equal weight** (1/N)
- **Inverse volatility** (weight inversely proportional to historical vol)

Position-size constraints from `config/settings.yaml` are enforced
automatically (max weight 20%, min weight 0%).

In [ ]:
# Compute returns for all tickers
closes = pd.DataFrame({t: clean_data[t]["Close"] for t in TICKERS})
returns = closes.pct_change().dropna()

print(f"Returns matrix: {returns.shape}")
print(f"Date range: {returns.index.min().date()} to {returns.index.max().date()}")

# Run optimisation methods
optimiser = PortfolioOptimiser(config=config)
mv_weights = optimiser.optimise(returns, method="mean_variance")
ew_weights = equal_weight(returns)
iv_weights = inverse_volatility(returns)

# Compare weights
weights_df = pd.DataFrame({
    "Mean-Variance": mv_weights,
    "Equal Weight": ew_weights,
    "Inverse Volatility": iv_weights,
})
print("\nPortfolio Weights:")
weights_df.round(4)

In [ ]:
# Target weights as a bar chart
fig, ax = plt.subplots(figsize=(10, 5))
weights_df.plot(kind="bar", ax=ax, alpha=0.85)
ax.set_title("Portfolio Weights by Optimisation Method")
ax.set_ylabel("Weight")
ax.set_xlabel("Ticker")
ax.axhline(y=1.0 / len(TICKERS), color="grey", linestyle="--", alpha=0.5,
           label=f"Equal = {1.0/len(TICKERS):.2f}")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation report highlighting risky pairs
corr_rpt = correlation_report(returns, threshold=0.7)

print("Asset Return Correlations:")
if corr_rpt["high_pairs"]:
    print("  Highly correlated pairs (>0.7):")
    for a, b, c in corr_rpt["high_pairs"]:
        print(f"    {a} <-> {b}: {c:.3f}")
else:
    print("  No pairs exceed the 0.7 correlation threshold.")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_rpt["correlation_matrix"], annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Asset Return Correlations")
plt.tight_layout()
plt.show()

---
## 6. Backtesting

Run three built-in strategies on a single ticker using `BacktestEngine`:

- **MeanReversion** \u2014 buy when RSI < 30, sell when RSI > 70
- **Momentum** \u2014 buy when Close > SMA-50, sell when Close < SMA-50
- **MACDCrossover** \u2014 buy/sell on MACD histogram zero-line crossover

Transaction costs (commission + slippage) are applied from
`config/settings.yaml`.

In [ ]:
# Use first ticker for single-asset backtesting
bt_ticker = TICKERS[0]
bt_prices = clean_data[bt_ticker]
bt_features = enriched[bt_ticker]

# Create strategies
strategies = [
    MeanReversionStrategy(),
    MomentumStrategy(),
    MACDCrossoverStrategy(),
]

# Compare all strategies
engine = BacktestEngine(config=config)
comparison_table = engine.compare(strategies, bt_prices, bt_features)
print(f"Strategy Comparison \u2014 {bt_ticker}:")
comparison_table

In [ ]:
# Run individual backtests to collect equity curves
results = {}
for s in strategies:
    results[s.name] = engine.run(s, bt_prices, bt_features)

# Plot equity curves overlaid
fig, ax = plt.subplots(figsize=(14, 5))
for name, res in results.items():
    ax.plot(res.equity_curve.index, res.equity_curve.values, label=name, linewidth=1)
ax.axhline(y=engine.initial_capital, color="grey", linestyle="--", alpha=0.5, label="Initial capital")
ax.set_title(f"Equity Curves \u2014 {bt_ticker}")
ax.set_ylabel("Portfolio Value (\u00a3)")
ax.set_xlabel("Date")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown chart for the best strategy (by Sharpe)
best_strat_name = comparison_table.iloc[0]["strategy_name"]
best_result = results[best_strat_name]

wealth = best_result.equity_curve
running_max = wealth.cummax()
drawdown = (wealth - running_max) / running_max

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(drawdown.index, drawdown.values, 0, color="red", alpha=0.3)
ax.plot(drawdown.index, drawdown.values, color="red", linewidth=0.8)
ax.set_title(f"Drawdown \u2014 {best_strat_name}")
ax.set_ylabel("Drawdown")
ax.set_xlabel("Date")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Full risk summary for the best strategy
print(f"Risk Summary \u2014 {best_strat_name}:\n")
for key, value in best_result.metrics.items():
    if isinstance(value, float):
        print(f"  {key:.<30s} {value:>10.4f}")
    else:
        print(f"  {key:.<30s} {str(value):>10s}")

---
## 7. Paper Trading Simulation

Demonstrate the execution layer by instantiating a `PaperBroker`,
computing rebalance orders from zero to the optimised portfolio weights,
executing the plan, and running reconciliation.

**Safety:** The default mode is always paper trading.  Live trading
requires config changes AND command-line flags AND typed confirmation.

In [ ]:
# Create and connect a paper broker
broker = PaperBroker(config=config)
broker.connect()

print(f"Broker connected: {broker.is_connected()}")
print(f"Initial cash: \u00a3{broker.cash:,.0f}")
print(f"Initial positions: {broker.get_positions()}")

# Use mean-variance weights from the portfolio optimisation step
target_weights = mv_weights
account_value = broker.cash

# Current prices: last close for each ticker
current_prices = {t: float(clean_data[t]["Close"].iloc[-1]) for t in TICKERS}
print(f"\nCurrent prices:")
for t, p in current_prices.items():
    print(f"  {t}: \u00a3{p:.2f}")

In [ ]:
# Compute rebalance orders from zero to target weights
oms = OrderManagementSystem(broker, config=config)
orders = oms.compute_rebalance_orders(
    target_weights, broker.get_positions(), account_value, current_prices,
)

# Show the order plan
print(f"Rebalance Plan ({len(orders)} orders):")
print(f"{'':>2s}{'Side':<6s} {'Qty':>8s} {'Ticker':<12s} {'Reason'}")
print("  " + "-" * 66)
for order in orders:
    print(f"  {order.side.upper():<6s} {order.quantity:>8g} {order.ticker:<12s} {order.reason}")

In [ ]:
# Execute the plan on the paper broker (not a dry run)
report = oms.execute_plan(orders, dry_run=False)

print("Execution Report:")
print(f"  Mode:            {report.mode}")
print(f"  Timestamp:       {report.timestamp.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"  Orders planned:  {len(report.orders_planned)}")
print(f"  Orders executed: {len(report.orders_executed)}")
print(f"  Orders failed:   {len(report.orders_failed)}")

In [ ]:
# Reconcile: compare target vs actual positions
recon = oms.reconcile(target_weights, account_value, current_prices)

print("Reconciliation:")
print(f"  Aligned:     {recon['aligned']}")
print(f"  Total drift: {recon['total_drift']:.4f}")
if recon["discrepancies"]:
    print(f"  Discrepancies ({len(recon['discrepancies'])}):")
    for d in recon["discrepancies"]:
        print(f"    {d['ticker']}: target={d['target_weight']:.4f}, "
              f"actual={d['actual_weight']:.4f}, diff={d['diff']:+.4f}")

# Show final positions and account value
print(f"\nFinal Positions:")
for t, qty in broker.get_positions().items():
    value = qty * current_prices.get(t, 0)
    print(f"  {t}: {qty:g} shares (\u00a3{value:,.0f})")
print(f"\nCash remaining: \u00a3{broker.cash:,.0f}")

broker.disconnect()
print(f"\nBroker disconnected: {not broker.is_connected()}")

---
## 8. Summary

This notebook demonstrated the **complete Quant Stack pipeline**:

| Stage | What happened |
|-------|---------------|
| **Data** | Generated 5 years of synthetic OHLCV for 5 UK large-caps, cleaned with `DataCleaner` |
| **Features** | Computed 30+ technical indicators via `FeaturePipeline` |
| **Models** | Compared RandomForest vs GradientBoosting with walk-forward CV (5 folds, gap=5) |
| **Portfolio** | Optimised weights using mean-variance, equal-weight, and inverse-volatility |
| **Backtest** | Tested MeanReversion, Momentum, and MACDCrossover strategies with realistic costs |
| **Execution** | Simulated a full rebalance cycle: order plan, paper execution, reconciliation |

### Next steps

1. **Real data** \u2014 Switch the data source from `synthetic` to `yfinance` in
   `config/settings.yaml` and run `python -m scripts.fetch_data`
2. **Custom strategies** \u2014 Subclass `Strategy` in `src/backtest/strategy.py`
   and register it with `strategy_registry`
3. **ML-driven signals** \u2014 Use the trained model's predictions as alpha
   signals for `PortfolioOptimiser.optimise(expected_returns=...)`
4. **Paper trading** \u2014 Run `python -m scripts.run_rebalance --execute` to
   execute trades on the paper broker
5. **Live trading** \u2014 Requires `--execute --live` flags, config change, and
   typed confirmation \u2014 safety gates prevent accidental live orders